In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/luongdiem/cold-scenario/cold_items_test.json
/kaggle/input/datasets/luongdiem/cold-scenario/cold_items_val.json
/kaggle/input/datasets/luongdiem/cold-scenario/test_cold.parquet
/kaggle/input/datasets/luongdiem/cold-scenario/train_cold.parquet
/kaggle/input/datasets/luongdiem/cold-scenario/val_cold.parquet
/kaggle/input/datasets/luongdiem/evaluator/evaluator.py
/kaggle/input/datasets/luongdiem/lightgcn-backbone-cold/g_i.npy
/kaggle/input/datasets/luongdiem/lightgcn-backbone-cold/g_u.npy
/kaggle/input/datasets/luongdiem/lightgcn-backbone-cold/item_index.json
/kaggle/input/datasets/luongdiem/lightgcn-backbone-cold/user_index_cold.json
/kaggle/input/datasets/luongdiem/course-llm-embeddings/course_dup_groups.parquet
/kaggle/input/datasets/luongdiem/course-llm-embeddings/embeddings_course_index.json
/kaggle/input/datasets/luongdiem/course-llm-embeddings/llm_profile_embeddings.npy
/kaggle/input/datasets/luongdiem/course-llm-embeddings/llm_profile_no_prereq_followup_embe

In [2]:
%%writefile coldrec_adapter.py
import copy
import hashlib
import json
import os
import pickle
import sys
import time
import numpy as np
import pandas as pd
import torch

ITEM_IDX_PATH = "/kaggle/input/datasets/luongdiem/lightgcn-backbone-cold/item_index.json"
USER_IDX_PATH = "/kaggle/input/datasets/luongdiem/lightgcn-backbone-cold/user_index_cold.json"
GU_PATH = "/kaggle/input/datasets/luongdiem/lightgcn-backbone-cold/g_u.npy"
GI_PATH = "/kaggle/input/datasets/luongdiem/lightgcn-backbone-cold/g_i.npy"

RAW_IDX_PATH = "/kaggle/input/datasets/luongdiem/course-llm-embeddings/raw_embeddings_course_index.json"
LLM_IDX_PATH = "/kaggle/input/datasets/luongdiem/course-llm-embeddings/embeddings_course_index.json"
RAW_NPY = "/kaggle/input/datasets/luongdiem/course-llm-embeddings/raw_course_text_embeddings.npy"
LLM_NPY = "/kaggle/input/datasets/luongdiem/course-llm-embeddings/llm_profile_embeddings.npy"
LLM_3D_NPY = "/kaggle/input/datasets/luongdiem/course-llm-embeddings/llm_profile_no_prereq_followup_embeddings.npy"

COLD_VAL_JSON = "/kaggle/input/datasets/luongdiem/cold-scenario/cold_items_val.json"
COLD_TEST_JSON = "/kaggle/input/datasets/luongdiem/cold-scenario/cold_items_test.json"
TRAIN_PARQUET = "/kaggle/input/datasets/luongdiem/cold-scenario/train_cold.parquet"
VAL_PARQUET = "/kaggle/input/datasets/luongdiem/cold-scenario/val_cold.parquet"
TEST_PARQUET = "/kaggle/input/datasets/luongdiem/cold-scenario/test_cold.parquet"

OUTPUT_DIR = "/kaggle/working"


def find_file_in_kaggle(filename, default_path=None):
    """Tự động tìm đường dẫn tệp trong /kaggle/input hoặc /kaggle/working."""
    if default_path and os.path.exists(default_path):
        return default_path
    for base_dir in ("/kaggle/input", "/kaggle/working", "."):
        if not os.path.exists(base_dir):
            continue
        for root, _, files in os.walk(base_dir):
            if filename in files:
                return os.path.join(root, filename)
    raise FileNotFoundError(
        f"Không tìm thấy tệp '{filename}' trong /kaggle/input! Hãy kiểm tra"
        " Dataset đã gắn vào Notebook."
    )


# Đường dẫn tệp phân nhóm trùng lặp P-RQ3 (tự động tìm nếu đặt ở thư mục khác)
DUP_GROUPS_PARQUET = find_file_in_kaggle(
    "course_dup_groups.parquet",
    default_path="/kaggle/input/datasets/luongdiem/course-llm-embeddings/course_dup_groups.parquet",
)

# Tự động tìm và import evaluator.py (hoặc evaluate.py)
try:
    eval_file_path = find_file_in_kaggle("evaluator.py")
    sys.path.insert(0, os.path.dirname(eval_file_path))
    from evaluator import evaluate, topk_random_ties
except FileNotFoundError:
    eval_file_path = find_file_in_kaggle("evaluate.py")
    sys.path.insert(0, os.path.dirname(eval_file_path))
    from evaluate import evaluate, topk_random_ties


def md5_hash(filepath):
    """Tính mã băm MD5 của tệp để truy vết phiên bản"""
    return hashlib.md5(open(filepath, "rb").read()).hexdigest()


def to_bool_mask(ids, c2i, n_items=3769):
    """Chuyển danh sách course_id sang mặt nạ bool dài 3.769"""
    m = torch.zeros(n_items, dtype=torch.bool)
    m[[c2i[str(c)] for c in ids]] = True
    return m


def build_hist(df, u2i, c2i):
    """Dựng từ điển lịch sử tương tác {u_idx: [i_idx, ...]}."""
    u_mapped = df["user_id"].astype(str).map(u2i)
    i_mapped = df["course_id"].astype(str).map(c2i)
    temp = pd.DataFrame({"u": u_mapped, "i": i_mapped}).dropna()
    temp["u"] = temp["u"].astype(int)
    temp["i"] = temp["i"].astype(int)
    return temp.groupby("u")["i"].apply(list).to_dict()


def _extract_id_list(json_path, key_name):
    data = json.load(open(json_path))
    items = data[key_name] if isinstance(data, dict) else data
    return [str(x) for x in items]


# Lớp bộ chuyển đổi dữ liệu và kiểm tra giao thức
class ColdRecDataAdapter:

    def __init__(self, output_dir=OUTPUT_DIR):
        self.output_dir = output_dir
        os.makedirs(self.output_dir, exist_ok=True)
        self._load_and_verify_all()

    def _load_and_verify_all(self):
        print("=== KIỂM TRA MD5 VÀ BẢNG ÁNH XẠ CHỈ SỐ CHUNG ===")
        # Đọc và đối chiếu bảng ánh xạ chỉ số chung 
        self.course_ids = _extract_id_list(ITEM_IDX_PATH, "course_id")
        self.users = _extract_id_list(USER_IDX_PATH, "user_id")

        assert len(self.course_ids) == 3769 and self.course_ids == sorted(self.course_ids), "Lỗi bảng item_index.json!"
        assert len(self.users) == 98220 and self.users == sorted(self.users), "Lỗi bảng user_index_cold.json!"

        if os.path.exists(RAW_IDX_PATH):
            raw_cids = _extract_id_list(RAW_IDX_PATH, "course_id")
            assert self.course_ids == raw_cids, "item_index.json không khớp raw_embeddings_course_index.json!"
        if os.path.exists(LLM_IDX_PATH):
            llm_cids = _extract_id_list(LLM_IDX_PATH, "course_id")
            assert self.course_ids == llm_cids, "item_index.json không khớp embeddings_course_index.json!"

        self.item2idx = {c: i for i, c in enumerate(self.course_ids)}
        self.user2idx = {u: i for i, u in enumerate(self.users)}
        self.n_users = len(self.users)
        self.n_items = len(self.course_ids)

        # Đọc danh sách khóa lạnh (251 cold-val, 502 cold-test)
        self.cold_val_ids = [str(c) for c in json.load(open(COLD_VAL_JSON))]
        self.cold_test_ids = [str(c) for c in json.load(open(COLD_TEST_JSON))]
        assert len(self.cold_val_ids) == 251 and len(self.cold_test_ids) == 502

        self.cold_val_idx = sorted([self.item2idx[c] for c in self.cold_val_ids])
        self.cold_test_idx = sorted([self.item2idx[c] for c in self.cold_test_ids])
        self.cold_idx = np.array(sorted(self.cold_val_idx + self.cold_test_idx), dtype=np.int64)
        assert len(self.cold_idx) == 753

        # Đọc các tệp tương tác và chạy assert chống rò rỉ
        self.tr_cold = pd.read_parquet(TRAIN_PARQUET)
        self.va_cold = pd.read_parquet(VAL_PARQUET)
        self.te_cold = pd.read_parquet(TEST_PARQUET)

        tr_items = set(self.tr_cold["course_id"].astype(str))
        cv, ct = set(self.cold_val_ids), set(self.cold_test_ids)
        assert not (cv & tr_items) and not (ct & tr_items) and not (cv & ct), "Rò rỉ khóa lạnh vào train hoặc trùng val/test!"
        assert set(self.va_cold["course_id"].astype(str)) <= cv
        assert set(self.te_cold["course_id"].astype(str)) <= ct
        assert set(self.va_cold["user_id"].astype(str)) <= set(self.tr_cold["user_id"].astype(str))
        assert set(self.te_cold["user_id"].astype(str)) <= set(self.tr_cold["user_id"].astype(str))

        # Xác định đúng 2.401 khóa ấm có tương tác trong train_cold
        self.warm_active_idx = np.unique(
            self.tr_cold["course_id"]
            .astype(str)
            .map(self.item2idx)
            .to_numpy(dtype=np.int64)
        )
        assert len(self.warm_active_idx) == 2401 and not (set(self.warm_active_idx.tolist()) & set(self.cold_idx.tolist()))

        # Đọc và kiểm tra Backbone LightGCN (g_u, g_i)
        self.g_u = np.load(GU_PATH).astype(np.float32)
        self.g_i = np.load(GI_PATH).astype(np.float32)

        gu_md5, gi_md5 = md5_hash(GU_PATH), md5_hash(GI_PATH)
        print(f"MD5 g_u.npy: {gu_md5} (Kỳ vọng: 7a170558...)")
        print(f"MD5 g_i.npy: {gi_md5} (Kỳ vọng: d17ad9df...)")

        assert self.g_u.shape == (98220, 64) and self.g_i.shape == (3769, 64)
        assert np.allclose(self.g_i[self.cold_idx], 0.0), "Khóa lạnh (753 khóa) phải có g_i = 0!"

        # Đảm bảo 615 khóa ấm không có tương tác trong train_cold cũng có g_i = 0
        inactive_mask = np.ones(self.n_items, dtype=bool)
        inactive_mask[self.warm_active_idx] = False
        assert np.allclose(self.g_i[inactive_mask], 0.0), "615 khóa ấm không tương tác phải có g_i = 0!"

        # Đọc và chuẩn hóa L2 cho các biến thể đặc trưng nội dung (raw, llm, fused, llm_3d, fused_3d)
        print(f"MD5 raw_emb: {md5_hash(RAW_NPY)} | MD5 llm_emb: {md5_hash(LLM_NPY)}")

        nz = lambda X: X / np.linalg.norm(X, axis=1, keepdims=True).clip(1e-9)
        self.S_raw = nz(np.load(RAW_NPY).astype(np.float32))
        self.S_llm = nz(np.load(LLM_NPY).astype(np.float32))
        assert self.S_raw.shape == (3769, 384) and self.S_llm.shape == (3769,384,)
        self.S_fused = np.concatenate([self.S_raw, self.S_llm], axis=1).astype(np.float32)  # [3769, 768]

        self.contents = {
            "raw": self.S_raw,
            "llm": self.S_llm,
            "fused": self.S_fused,
        }

        if os.path.exists(LLM_3D_NPY):
            print(f"MD5 llm_3d_emb: {md5_hash(LLM_3D_NPY)}")
            self.S_llm_3d = nz(np.load(LLM_3D_NPY).astype(np.float32))
            assert self.S_llm_3d.shape == (3769, 384)
            self.S_fused_3d = np.concatenate(
                [self.S_raw, self.S_llm_3d], axis=1
            ).astype(np.float32)
            self.contents["llm_3d"] = self.S_llm_3d
            self.contents["fused_3d"] = self.S_fused_3d

        # Dựng từ điển mặt nạ lịch sử, đích đánh giá và mặt nạ ứng viên (a), (b)
        self.hist_tr = build_hist(self.tr_cold, self.user2idx, self.item2idx)
        self.hist_trva = build_hist(
            pd.concat([self.tr_cold, self.va_cold], ignore_index=True),
            self.user2idx,
            self.item2idx,
        )
        self.val_target_dict = build_hist(
            self.va_cold, self.user2idx, self.item2idx
        )
        self.test_target_dict = build_hist(
            self.te_cold, self.user2idx, self.item2idx
        )
        assert (
            len(self.val_target_dict) == 2092
            and len(self.test_target_dict) == 3340
        )

        self.cand_val_a = to_bool_mask(
            self.cold_val_ids, self.item2idx, self.n_items
        )
        self.cand_test_a = to_bool_mask(
            self.cold_test_ids, self.item2idx, self.n_items
        )
        self.cand_test_b = torch.ones(self.n_items, dtype=torch.bool)

        # Phân nhóm P-RQ3 ("trung" vs "moi") từ course_dup_groups.parquet
        dup_df = pd.read_parquet(DUP_GROUPS_PARQUET)
        c2dup = dict(zip(dup_df["course_id"].astype(str), dup_df["dup_group_id"]))
        active_warm_course_ids = [self.course_ids[i] for i in self.warm_active_idx]
        warm_active_dup_groups = {c2dup[c] for c in active_warm_course_ids}

        self.cold_test_dup_ids = [c for c in self.cold_test_ids if c2dup[c] in warm_active_dup_groups]
        self.cold_test_new_ids = [
            c
            for c in self.cold_test_ids
            if c2dup[c] not in warm_active_dup_groups
        ]
        assert (
            len(self.cold_test_dup_ids) == 124
            and len(self.cold_test_new_ids) == 378
        ), (
            f"Sai số lượng khóa trùng/mới: {len(self.cold_test_dup_ids)} /"
            f" {len(self.cold_test_new_ids)}"
        )

        self.groups_prq3 = {
            "trung": {self.item2idx[c] for c in self.cold_test_dup_ids},
            "moi": {self.item2idx[c] for c in self.cold_test_new_ids},
        }
        print("=== HOÀN TẤT KIỂM TRA: MỌI ASSERT ĐỀU ĐẠT CHUẨN! ===\n")

    def export_coldrec_files(
        self, dataset_name="MOOCCubeX", content_variant="raw"
    ):
        """Xuất bộ dữ liệu sang đúng cấu trúc thư mục mà ColdRec (ALDI, CGRC, USIM)

        và mã chính thức zfnWong/ALDI yêu cầu, giữ nguyên chỉ số 0..98219 và
        0..3768.
        """
        ds_dir = os.path.join(self.output_dir, "ColdRec_Data", dataset_name)
        os.makedirs(ds_dir, exist_ok=True)

        # Ánh xạ các bảng tương tác sang chỉ số nguyên (user, item)
        def to_mapped_df(df):
            u = df["user_id"].astype(str).map(self.user2idx).astype(np.int64)
            i = df["course_id"].astype(str).map(self.item2idx).astype(np.int64)
            return pd.DataFrame({"user": u, "item": i})

        tr_df = to_mapped_df(self.tr_cold)
        va_df = to_mapped_df(self.va_cold)
        te_df = to_mapped_df(self.te_cold)

        # Lưu các tệp CSV tương tác cho ColdRec
        tr_df.to_csv(os.path.join(ds_dir, "warm_train.csv"), index=False)
        va_df.to_csv(os.path.join(ds_dir, "cold_item_val.csv"), index=False)
        te_df.to_csv(os.path.join(ds_dir, "cold_item_test.csv"), index=False)

        # Lưu danh sách ID khóa ấm có tương tác (2.401 khóa) và khóa lạnh
        np.save(os.path.join(ds_dir, "warm_item_ids.npy"), self.warm_active_idx)
        np.save(
            os.path.join(ds_dir, "cold_val_item_ids.npy"),
            np.array(self.cold_val_idx, dtype=np.int64),
        )
        np.save(
            os.path.join(ds_dir, "cold_test_item_ids.npy"),
            np.array(self.cold_test_idx, dtype=np.int64),
        )
        np.save(os.path.join(ds_dir, "cold_item_ids.npy"), self.cold_idx)

        # Lưu Pre-trained Teacher Embeddings (LightGCN nền chung)
        warm_emb_concat = np.concatenate([self.g_u, self.g_i], axis=0)  # [98220 + 3769, 64]
        np.save(
            os.path.join(ds_dir, f"{dataset_name}_lightgcn.npy"),
            warm_emb_concat,
        )
        np.save(os.path.join(ds_dir, "warm_emb.npy"), warm_emb_concat)
        np.save(os.path.join(ds_dir, "g_u.npy"), self.g_u)
        np.save(os.path.join(ds_dir, "g_i.npy"), self.g_i)

        # Lưu ma trận đặc trưng nội dung khóa học theo biến thể được chọn ('raw', 'llm', 'fused')
        content_mat = self.contents[content_variant]
        np.save(
            os.path.join(ds_dir, f"{dataset_name}_item_content.npy"),
            content_mat,
        )
        np.save(os.path.join(ds_dir, "item_content.npy"), content_mat)

        # Lưu từ điển thông tin tổng hợp (para_dict.pkl) cho bộ nạp của ColdRec / ALDI
        para_dict = {
            "user_num": self.n_users,
            "item_num": self.n_items,
            "warm_items": set(self.warm_active_idx.tolist()),
            "cold_val_items": set(self.cold_val_idx),
            "cold_test_items": set(self.cold_test_idx),
            "cold_items": set(self.cold_idx.tolist()),
            "emb_dim": 64,
            "content_dim": int(content_mat.shape[1]),
            "user_his": self.hist_tr,
            "val_user_his": self.hist_trva,
        }
        with open(os.path.join(ds_dir, "para_dict.pkl"), "wb") as f:
            pickle.dump(para_dict, f)

        print(
            f"[OK] Đã xuất dữ liệu ColdRec ({content_variant},"
            f" dim={content_mat.shape[1]}) tại: {ds_dir}"
        )
        return ds_dir


# Các hàm đánh giá của nhóm (tập a, b, P-RQ3, P-RQ4)
@torch.no_grad()
def eval_subgroups(
    score_fn, targets, mask, cand, groups, ks=(5, 10, 20), tie_seed=0, bs=1024
):
    """Hàm tính P-RQ3 đúng chuẩn:

    Xếp hạng trên TOÀN BỘ 502 khóa của tập (a), sau đó chỉ tính chỉ số trên đích
    thuộc từng nhóm.
    """
    users, K = sorted(targets), max(ks)
    g = torch.Generator().manual_seed(tie_seed)
    disc = 1.0 / np.log2(np.arange(2, K + 2))
    cand = torch.as_tensor(cand)
    out = {
        n: {f"{m}@{k}": [] for m in ("Recall", "NDCG", "HR") for k in ks}
        for n in groups
    }
    for s in range(0, len(users), bs):
        ub = users[s : s + bs]
        sc = score_fn(torch.tensor(ub)).float().cpu().clone()
        sc[:, ~cand] = -float("inf")
        for r, u in enumerate(ub):
            if mask.get(u):
                sc[r, list(mask[u])] = -float("inf")
        top = topk_random_ties(sc, K, g)
        for r, u in enumerate(ub):
            for n, gset in groups.items():
                gt = [i for i in targets[u] if i in gset]
                if not gt:
                    continue
                hit = np.isin(top[r], gt).astype(float)
                for k in ks:
                    h = hit[:k]
                    idcg = disc[: min(len(gt), k)].sum()
                    out[n][f"Recall@{k}"].append(h.sum() / len(gt))
                    out[n][f"NDCG@{k}"].append((h * disc[:k]).sum() / idcg)
                    out[n][f"HR@{k}"].append(float(h.sum() > 0))
    return {n: {m: np.array(v) for m, v in d.items()} for n, d in out.items()}


@torch.no_grad()
def evaluate_coldrec_embeddings(
    adapter, u_emb, i_emb, model_name, variant_name, seed, device="cuda:0"
):
    """Hàm cầu nối bắt buộc:

    Nhận u_emb [98220, 64] và i_emb [3769, 64] xuất ra từ mô hình ColdRec (ALDI
    / CGRC / USIM), chấm điểm bằng evaluator.py và lưu đầy đủ file phục
    vụ Wilcoxon + P-RQ3.
    """
    u_t = torch.as_tensor(u_emb, dtype=torch.float32, device=device)
    i_t = torch.as_tensor(i_emb, dtype=torch.float32, device=device)
    assert u_t.shape == (98220, 64) and i_t.shape == (3769, 64)

    score_fn = lambda ub: (u_t[ub.to(device)] @ i_t.t()).cpu()

    # Đánh giá trên tập (a) (502 khóa cold-test) và tập (b) (3.769 khóa)
    kw = dict(mask=adapter.hist_trva, ks=(5, 10, 20), tie_seed=seed)
    res_a = evaluate(score_fn, adapter.test_target_dict, cand=adapter.cand_test_a, **kw)
    res_b = evaluate(score_fn, adapter.test_target_dict, cand=adapter.cand_test_b, **kw)

    # Đánh giá P-RQ3 theo nhóm khóa trùng / mới
    res_prq3 = eval_subgroups(
        score_fn,
        adapter.test_target_dict,
        adapter.hist_trva,
        adapter.cand_test_a,
        adapter.groups_prq3,
        ks=(5, 10, 20),
        tie_seed=seed,
    )
    assert (
        len(res_prq3["trung"]["Recall@20"]) == 1028
        and len(res_prq3["moi"]["Recall@20"]) == 2844
    )

    # Lưu biểu diễn người học và khóa học của seed này
    rep_dir = os.path.join(adapter.output_dir, f"results_{model_name}", "item_reps")
    os.makedirs(rep_dir, exist_ok=True)
    np.save(
        os.path.join(rep_dir, f"{model_name}_{variant_name}_seed{seed}.npy"),
        i_t.cpu().numpy(),
    )
    np.save(
        os.path.join(rep_dir, f"u_e_{variant_name}_seed{seed}.npy"),
        u_t.cpu().numpy(),
    )
    np.save(
        os.path.join(rep_dir, f"v_e_{variant_name}_seed{seed}.npy"),
        i_t.cpu().numpy(),
    )

    return res_a, res_b, res_prq3


if __name__ == "__main__":
    # Khởi tạo Adapter, chạy toàn bộ kiểm tra assert và xuất dữ liệu cho cả 5 biến thể (gồm 3D cho E-RQ3)
    adapter = ColdRecDataAdapter()
    for var in ["raw", "llm", "fused", "llm_3d", "fused_3d"]:
        if var in adapter.contents:
            adapter.export_coldrec_files(
                dataset_name=f"MOOCCubeX_{var}", content_variant=var
            )

Writing coldrec_adapter.py


In [3]:
!python coldrec_adapter.py

=== KIỂM TRA MD5 VÀ BẢNG ÁNH XẠ CHỈ SỐ CHUNG ===
MD5 g_u.npy: 7a170558642de73d301970e8ce7f0a86 (Kỳ vọng: 7a170558...)
MD5 g_i.npy: d17ad9dfc4f4d6cb6de915c8649070d7 (Kỳ vọng: d17ad9df...)
MD5 raw_emb: f80445649ef8a8217d048bd3e7ac0ad5 | MD5 llm_emb: fc1819307be227a1544e1be0a4017393
MD5 llm_3d_emb: 1af436095463ad0af3ef1203ec0ebd6c
=== HOÀN TẤT KIỂM TRA: MỌI ASSERT ĐỀU ĐẠT CHUẨN! ===

[OK] Đã xuất dữ liệu ColdRec (raw, dim=384) tại: /kaggle/working/ColdRec_Data/MOOCCubeX_raw
[OK] Đã xuất dữ liệu ColdRec (llm, dim=384) tại: /kaggle/working/ColdRec_Data/MOOCCubeX_llm
[OK] Đã xuất dữ liệu ColdRec (fused, dim=768) tại: /kaggle/working/ColdRec_Data/MOOCCubeX_fused
[OK] Đã xuất dữ liệu ColdRec (llm_3d, dim=384) tại: /kaggle/working/ColdRec_Data/MOOCCubeX_llm_3d
[OK] Đã xuất dữ liệu ColdRec (fused_3d, dim=768) tại: /kaggle/working/ColdRec_Data/MOOCCubeX_fused_3d


In [4]:
!git clone https://github.com/YuanchenBei/ColdRec.git
!git clone https://github.com/zfnWong/ALDI.git
!cd ColdRec && git rev-parse HEAD  # Ghi lại commit hash 

Cloning into 'ColdRec'...
remote: Enumerating objects: 561, done.
remote: Counting objects: 100% (101/101), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 561 (delta 69), reused 67 (delta 51), pack-reused 460 (from 1)
Receiving objects: 100% (561/561), 2.36 MiB | 17.54 MiB/s, done.
Resolving deltas: 100% (359/359), done.
Cloning into 'ALDI'...
remote: Enumerating objects: 51, done.
remote: Counting objects: 100% (51/51), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 51 (delta 12), reused 45 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (51/51), 38.63 MiB | 26.95 MiB/s, done.
Resolving deltas: 100% (12/12), done.
1108192665aca2cb94b3aeb5bfed29850874eb1c


In [5]:
%%writefile run_aldi.py
import copy
import itertools
import json
import os
import random
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from coldrec_adapter import ColdRecDataAdapter, evaluate, eval_subgroups

DEV = "cuda:0" if torch.cuda.is_available() else "cpu"
SEEDS = [42, 2023, 2024, 1007, 999]
ROBUST_VAL_SEEDS = [42, 2023, 2024]


# Kiến trúc ALDI chuyển nguyên bản từ mã TensorFlow chính thức 
class ALDI(nn.Module):
    def __init__(self, d_content, d_emb=64, h=200):
        super().__init__()
        self.f_I = nn.Sequential(
            nn.Linear(d_content, h),
            nn.BatchNorm1d(h),
            nn.Tanh(),
            nn.Linear(h, d_emb),
        )
        self.f_U = nn.Sequential(
            nn.Linear(d_emb, h),
            nn.BatchNorm1d(h),
            nn.Tanh(),
            nn.Linear(h, d_emb),
        )
        for m in self.modules():
            if isinstance(m, nn.Linear):  # truncated_normal(stddev=0.01)
                nn.init.trunc_normal_(m.weight, std=0.01, a=-0.02, b=0.02)
                nn.init.zeros_(m.bias)

    def l2(self):  # l2_regularizer: sum(w^2)/2, cả weight và bias
        return (
            sum(
                (m.weight ** 2).sum() + (m.bias ** 2).sum()
                for m in self.modules()
                if isinstance(m, nn.Linear)
            )
            / 2
        )


def aldi_loss(model, u_t, c_pos, c_neg, e_pos, e_neg, w_pos, cfg):
    s_pos, s_neg = model.f_I(torch.cat([c_pos, c_neg])).chunk(2)  # BN trên lô dương + âm
    u_s = model.f_U(u_t)
    sp, sn = (u_s * s_pos).sum(-1), (u_s * s_neg).sum(-1)         # học sinh
    tp, tn = (u_t * e_pos).sum(-1), (u_t * e_neg).sum(-1)         # giáo viên (LightGCN)
    bce = F.binary_cross_entropy_with_logits
    sup = bce(sp - sn, torch.ones_like(sp))
    rank = (w_pos * bce(sp - sn, torch.sigmoid(tp - tn), reduction="none")).mean()
    s_id = (s_pos * s_pos).sum(-1) - (s_pos @ s_neg.t()).mean(1)
    t_id = (e_pos * e_pos).sum(-1) - (e_pos @ e_neg.t()).mean(1)
    iden = (w_pos * bce(s_id, torch.sigmoid(t_id), reduction="none")).mean()
    dist = ((tp - sp).abs() + (tn - sn).abs()).mean()
    return (
        sup
        + cfg["alpha"] * rank
        + cfg["beta"] * iden
        + cfg["gamma"] * dist
        + cfg["reg"] * model.l2()
    )


# Khởi tạo Adapter, trọng số tần suất khóa (tws) và hàm lấy mẫu 
ad = ColdRecDataAdapter()
g_u_t = torch.tensor(ad.g_u, dtype=torch.float32, device=DEV)
g_i_t = torch.tensor(ad.g_i, dtype=torch.float32, device=DEV)

u_tr = ad.tr_cold.user_id.astype(str).map(ad.user2idx).to_numpy()
i_tr = ad.tr_cold.course_id.astype(str).map(ad.item2idx).to_numpy()
deg_u = np.bincount(u_tr, minlength=ad.n_users)
f = np.zeros(ad.n_items)
np.add.at(f, i_tr, 1.0 / deg_u[u_tr])
freq = np.ones(ad.n_items, dtype=np.float32)
freq[ad.warm_active_idx] = f[ad.warm_active_idx]
M = 4.0
a = M / (ad.n_users / ad.n_items)
w_item = torch.tensor(
    np.clip(np.tanh(a * freq), 0, np.tanh(M)), dtype=torch.float32, device=DEV
)

warm_users = np.array(sorted(ad.hist_tr))


def sample_epoch(rng):  # như utils.bpr_neg_samp
    users = rng.choice(warm_users, size=len(u_tr), replace=True)
    pos = np.array([ad.hist_tr[u][rng.integers(len(ad.hist_tr[u]))] for u in users])
    neg = rng.choice(ad.warm_active_idx, size=len(u_tr), replace=True)
    return users, pos, neg


# 1.368 khóa không có tương tác trong train (753 khóa lạnh + 615 khóa ấm không hoạt động)
gen_cols = torch.tensor(
    np.setdiff1d(np.arange(ad.n_items), ad.warm_active_idx), device=DEV
)


def aldi_score_fn(model, content_t):
    model.eval()
    with torch.no_grad():
        u_map, gen = model.f_U(g_u_t), model.f_I(content_t)

    def fn(ub):
        ub = ub.to(DEV)
        sc = g_u_t[ub] @ g_i_t.t()                     # khóa ấm: điểm giáo viên
        sc[:, gen_cols] = u_map[ub] @ gen[gen_cols].t() # khóa còn lại: học sinh
        return sc

    return fn


def train_aldi(content_t, cfg, seed, max_epochs=200, patience=10, bs=1024):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    rng = np.random.default_rng(seed)
    model = ALDI(content_t.shape[1]).to(DEV)
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"])  # L2 nằm trong loss
    w_t = w_item if cfg["tws"] else torch.ones_like(w_item)
    best, best_state, bad, best_ep = -1.0, None, 0, 0
    for ep in range(1, max_epochs + 1):
        model.train()
        users, pos, neg = (torch.tensor(x, device=DEV) for x in sample_epoch(rng))
        for s in range(0, len(users) - bs, bs):  # như main.py: bỏ lô cuối thiếu
            u, p, n = users[s : s + bs], pos[s : s + bs], neg[s : s + bs]
            loss = aldi_loss(
                model,
                g_u_t[u],
                content_t[p],
                content_t[n],
                g_i_t[p],
                g_i_t[n],
                w_t[p],
                cfg,
            )
            opt.zero_grad()
            loss.backward()
            opt.step()
        r = evaluate(
            aldi_score_fn(model, content_t),
            ad.val_target_dict,
            mask=ad.hist_tr,
            cand=ad.cand_val_a,
            ks=(20,),
            tie_seed=seed,
        )
        v = float(np.mean(r["NDCG@20"]))
        if v > best:
            best, best_state, bad, best_ep = (
                v,
                copy.deepcopy(model.state_dict()),
                0,
                ep,
            )
        else:
            bad += 1
            if bad > patience:
                break  # như main.py
    model.load_state_dict(best_state)
    return model, best, best_ep


# Không gian siêu tham số 20 cấu hình 
space = dict(
    lr=[1e-4, 5e-4, 1e-3, 5e-3],
    alpha=[0.1, 0.5, 0.9],
    beta=[0.01, 0.05, 0.1],
    gamma=[0.05, 0.1, 0.5],
    reg=[1e-5, 1e-4, 1e-3],
    tws=[0, 1],
)
grid = [dict(zip(space, v)) for v in itertools.product(*space.values())]
trials = random.Random(42).sample(grid, 20)
DEFAULT = dict(lr=1e-3, alpha=0.9, beta=0.05, gamma=0.1, reg=1e-4, tws=0)
if DEFAULT not in trials:
    trials[-1] = DEFAULT  # vẫn đúng 20 cấu hình


def evaluate_5_seeds(inp_name, cfg, tag_suffix=""):
    content_t = torch.tensor(ad.contents[inp_name], dtype=torch.float32, device=DEV)
    out_dir = os.path.join(ad.output_dir, "results_aldi")
    rep_dir = os.path.join(out_dir, "item_reps")
    score_dir = os.path.join(ad.output_dir, "scores")
    os.makedirs(rep_dir, exist_ok=True)
    os.makedirs(score_dir, exist_ok=True)

    metrics_a = {"Recall@20": [], "NDCG@20": []}
    metrics_b = {"Recall@20": [], "NDCG@20": []}
    sub_metrics = {
        "trung": {"Recall@20": [], "NDCG@20": []},
        "moi": {"Recall@20": [], "NDCG@20": []},
    }
    user_ndcg_a_seeds = []
    user_ndcg_sub_seeds = {"trung": [], "moi": []}

    n_params = sum(p.numel() for p in ALDI(content_t.shape[1]).parameters() if p.requires_grad)

    for s in SEEDS:
        t0 = time.time()
        model, val_ndcg, best_ep = train_aldi(content_t, cfg, seed=s)
        fn = aldi_score_fn(model, content_t)

        kw = dict(mask=ad.hist_trva, ks=(5, 10, 20), tie_seed=s)
        res_a = evaluate(fn, ad.test_target_dict, cand=ad.cand_test_a, **kw)
        res_b = evaluate(fn, ad.test_target_dict, cand=ad.cand_test_b, **kw)
        res_prq3 = eval_subgroups(
            fn,
            ad.test_target_dict,
            ad.hist_trva,
            ad.cand_test_a,
            ad.groups_prq3,
            ks=(5, 10, 20),
            tie_seed=s,
        )

        # Lưu vector biểu diễn theo từng seed (bọc trong torch.no_grad)
        model.eval()
        with torch.no_grad():
            u_map, gen = model.f_U(g_u_t), model.f_I(content_t)
        save_key = f"{inp_name}{tag_suffix}"
        np.save(os.path.join(rep_dir, f"u_e_{save_key}_seed{s}.npy"), u_map.detach().cpu().numpy())
        np.save(os.path.join(rep_dir, f"v_e_{save_key}_seed{s}.npy"), gen.detach().cpu().numpy())

        # Ghi nhận chỉ số
        metrics_a["Recall@20"].append(float(np.mean(res_a["Recall@20"])))
        metrics_a["NDCG@20"].append(float(np.mean(res_a["NDCG@20"])))
        metrics_b["Recall@20"].append(float(np.mean(res_b["Recall@20"])))
        metrics_b["NDCG@20"].append(float(np.mean(res_b["NDCG@20"])))

        user_ndcg_a_seeds.append(np.asarray(res_a["NDCG@20"], dtype=np.float32))
        for g in ("trung", "moi"):
            sub_metrics[g]["Recall@20"].append(float(np.mean(res_prq3[g]["Recall@20"])))
            sub_metrics[g]["NDCG@20"].append(float(np.mean(res_prq3[g]["NDCG@20"])))
            user_ndcg_sub_seeds[g].append(np.asarray(res_prq3[g]["NDCG@20"], dtype=np.float32))

        print(
            f"  [Seed {s}] ep={best_ep:02d} | val_NDCG@20={val_ndcg:.4f} | "
            f"(a) R@20={metrics_a['Recall@20'][-1]:.4f}, NDCG@20={metrics_a['NDCG@20'][-1]:.4f} | "
            f"(b) NDCG@20={metrics_b['NDCG@20'][-1]:.4f} | "
            f"trùng NDCG@20={sub_metrics['trung']['NDCG@20'][-1]:.4f}, "
            f"mới NDCG@20={sub_metrics['moi']['NDCG@20'][-1]:.4f} ({time.time()-t0:.1f}s)"
        )

    # Lưu tệp điểm từng học viên (trung bình qua 5 seed)
    save_key = f"{inp_name}{tag_suffix}"
    mean_user_a = np.mean(np.stack(user_ndcg_a_seeds, axis=0), axis=0)
    assert len(mean_user_a) == 3340
    np.save(os.path.join(score_dir, f"user_ndcg20_aldi_{save_key}.npy"), mean_user_a)

    for g, expected_len in (("trung", 1028), ("moi", 2844)):
        mean_user_g = np.mean(np.stack(user_ndcg_sub_seeds[g], axis=0), axis=0)
        assert len(mean_user_g) == expected_len
        np.save(os.path.join(score_dir, f"user_ndcg20_aldi_{save_key}_{g}.npy"), mean_user_g)

    summary = {
        "input": save_key,
        "cfg": cfg,
        "n_params": n_params,
        "a_R20_mean": float(np.mean(metrics_a["Recall@20"])),
        "a_R20_std": float(np.std(metrics_a["Recall@20"], ddof=1)),
        "a_NDCG20_mean": float(np.mean(metrics_a["NDCG@20"])),
        "a_NDCG20_std": float(np.std(metrics_a["NDCG@20"], ddof=1)),
        "b_R20_mean": float(np.mean(metrics_b["Recall@20"])),
        "b_NDCG20_mean": float(np.mean(metrics_b["NDCG@20"])),
        "trung_R20_mean": float(np.mean(sub_metrics["trung"]["Recall@20"])),
        "trung_NDCG20_mean": float(np.mean(sub_metrics["trung"]["NDCG@20"])),
        "moi_R20_mean": float(np.mean(sub_metrics["moi"]["Recall@20"])),
        "moi_NDCG20_mean": float(np.mean(sub_metrics["moi"]["NDCG@20"])),
    }
    print(
        f"=> TỔNG KẾT ALDI [{save_key}] (params={n_params:,}):\n"
        f"   Cold-test (a): NDCG@20 = {summary['a_NDCG20_mean']:.4f} ± {summary['a_NDCG20_std']:.4f} | "
        f"R@20 = {summary['a_R20_mean']:.4f} ± {summary['a_R20_std']:.4f}\n"
        f"   All-items (b): NDCG@20 = {summary['b_NDCG20_mean']:.4f} | R@20 = {summary['b_R20_mean']:.4f}\n"
        f"   Nhóm TRÙNG / MỚI: R@20 = {summary['trung_R20_mean']:.4f} / {summary['moi_R20_mean']:.4f} | "
        f"NDCG@20 = {summary['trung_NDCG20_mean']:.4f} / {summary['moi_NDCG20_mean']:.4f}\n"
    )
    return summary


# Ghi nhận sẵn kết quả 20 trials của 'raw' đã chạy xong ở lần trước để không tốn GPU chạy lại
PRECOMPUTED_TRIALS = {
    "raw": {
        "best_val": 0.1018,
        "best_ep": 25,
        "best_cfg": {"lr": 0.005, "alpha": 0.9, "beta": 0.01, "gamma": 0.1, "reg": 0.001, "tws": 0},
        "top3_cfgs": [
            {"lr": 0.005, "alpha": 0.9, "beta": 0.01, "gamma": 0.1, "reg": 0.001, "tws": 0},
            {"lr": 0.005, "alpha": 0.9, "beta": 0.05, "gamma": 0.05, "reg": 0.001, "tws": 0},
            {"lr": 0.0005, "alpha": 0.5, "beta": 0.01, "gamma": 0.1, "reg": 1e-05, "tws": 1},
        ],
    }
}


def tune_and_run_aldi(
    variants=("raw", "llm", "fused"),
    run_erq3=True,
    run_w6_robustness=False,
):
    out_dir = os.path.join(ad.output_dir, "results_aldi")
    os.makedirs(out_dir, exist_ok=True)
    summary_path = os.path.join(out_dir, "aldi_summary.json")
    all_summaries = {}
    best_cfgs = {}
    top3_cfgs = {}

    # Đã xác nhận DEFAULT (seed=42, raw) đạt 0.0773 tại epoch 5
    print("=== KIỂM TRA CẤU HÌNH MẶC ĐỊNH (DEFAULT, seed=42, raw) ===")
    print(
        "ALDI chính thức cấu hình DEFAULT (raw, seed=42): "
        "cold-val NDCG@20 = 0.0773 (tại epoch 5)\n"
    )

    # Tinh chỉnh 20 cấu hình (seed=42) và chạy 5 seed cho các đầu vào chính
    for inp in variants:
        if inp in PRECOMPUTED_TRIALS:
            info = PRECOMPUTED_TRIALS[inp]
            best_val, best_ep, best_cfg = info["best_val"], info["best_ep"], info["best_cfg"]
            best_cfgs[inp] = best_cfg
            top3_cfgs[inp] = info["top3_cfgs"]
            print(
                f"=== SỬ DỤNG KẾT QUẢ 20 TRIALS ĐÃ CHẠY CỦA ALDI [{inp}] ===\n"
                f"-> Cấu hình tốt nhất [{inp}]: val_NDCG@20={best_val:.4f} (ep={best_ep}) | {best_cfg}"
            )
        else:
            print(f"=== TINH CHỈNH 20 CẤU HÌNH CHO ALDI [{inp}] (seed=42) ===")
            content_t = torch.tensor(ad.contents[inp], dtype=torch.float32, device=DEV)
            trial_records = []
            for idx, cfg in enumerate(trials, 1):
                _, val_ndcg, best_ep = train_aldi(content_t, cfg, seed=42)
                trial_records.append((val_ndcg, best_ep, cfg))
                print(f"  Trial {idx:02d}/20 | val_NDCG@20={val_ndcg:.4f} (ep={best_ep:02d}) | {cfg}")

            trial_records.sort(key=lambda x: x[0], reverse=True)
            best_val, best_ep, best_cfg = trial_records[0]
            best_cfgs[inp] = best_cfg
            top3_cfgs[inp] = [r[2] for r in trial_records[:3]]
            print(f"-> Cấu hình tốt nhất [{inp}]: val_NDCG@20={best_val:.4f} (ep={best_ep}) | {best_cfg}")

        all_summaries[inp] = evaluate_5_seeds(inp, best_cfg)
        with open(summary_path, "w", encoding="utf-8") as f:
            json.dump(all_summaries, f, indent=2, ensure_ascii=False)

    # Hồ sơ 3 chiều dùng cấu hình của 5 chiều tương ứng (không tune lại)
    if run_erq3:
        for inp_3d, base_5d in (("llm_3d", "llm"), ("fused_3d", "fused")):
            if inp_3d in ad.contents and base_5d in best_cfgs:
                print(f"=== CHẠY ALDI [{inp_3d}] DÙNG CẤU HÌNH CỦA [{base_5d}] ===")
                all_summaries[inp_3d] = evaluate_5_seeds(inp_3d, best_cfgs[base_5d])
                with open(summary_path, "w", encoding="utf-8") as f:
                    json.dump(all_summaries, f, indent=2, ensure_ascii=False)

    # Phân tích độ bền chọn cấu hình bằng trung bình 3 seed trên top-3 trials
    if run_w6_robustness:
        for inp in variants:
            print(f"=== PHÂN TÍCH ĐỘ BỀN CHỌN 3 SEED CHO ALDI [{inp}] ===")
            content_t = torch.tensor(ad.contents[inp], dtype=torch.float32, device=DEV)
            best_3s_val, best_3s_cfg = -1.0, None
            for cfg in top3_cfgs[inp]:
                vals = [train_aldi(content_t, cfg, seed=s)[1] for s in ROBUST_VAL_SEEDS]
                m_val = float(np.mean(vals))
                print(f"  Top-3 candidate {cfg} -> 3-seed cold-val mean = {m_val:.4f}")
                if m_val > best_3s_val:
                    best_3s_val, best_3s_cfg = m_val, cfg
            print(f"-> Cấu hình chọn theo 3 seed [{inp}]: {best_3s_cfg} (val={best_3s_val:.4f})")
            all_summaries[f"{inp}_robust3s"] = evaluate_5_seeds(
                inp, best_3s_cfg, tag_suffix="_robust3s"
            )
            with open(summary_path, "w", encoding="utf-8") as f:
                json.dump(all_summaries, f, indent=2, ensure_ascii=False)


if __name__ == "__main__":
    tune_and_run_aldi(
        variants=("raw", "llm", "fused"),
        run_erq3=True,
        run_w6_robustness=True,  # Đặt True khi chạy phân tích độ bền 
    )

Writing run_aldi.py


In [6]:
!python run_aldi.py

=== KIỂM TRA MD5 VÀ BẢNG ÁNH XẠ CHỈ SỐ CHUNG ===
MD5 g_u.npy: 7a170558642de73d301970e8ce7f0a86 (Kỳ vọng: 7a170558...)
MD5 g_i.npy: d17ad9dfc4f4d6cb6de915c8649070d7 (Kỳ vọng: d17ad9df...)
MD5 raw_emb: f80445649ef8a8217d048bd3e7ac0ad5 | MD5 llm_emb: fc1819307be227a1544e1be0a4017393
MD5 llm_3d_emb: 1af436095463ad0af3ef1203ec0ebd6c
=== HOÀN TẤT KIỂM TRA: MỌI ASSERT ĐỀU ĐẠT CHUẨN! ===

=== KIỂM TRA CẤU HÌNH MẶC ĐỊNH (DEFAULT, seed=42, raw) ===
ALDI chính thức cấu hình DEFAULT (raw, seed=42): cold-val NDCG@20 = 0.0773 (tại epoch 5)

=== SỬ DỤNG KẾT QUẢ 20 TRIALS ĐÃ CHẠY CỦA ALDI [raw] ===
-> Cấu hình tốt nhất [raw]: val_NDCG@20=0.1018 (ep=25) | {'lr': 0.005, 'alpha': 0.9, 'beta': 0.01, 'gamma': 0.1, 'reg': 0.001, 'tws': 0}
  [Seed 42] ep=25 | val_NDCG@20=0.1018 | (a) R@20=0.1578, NDCG@20=0.0720 | (b) NDCG@20=0.0243 | trùng NDCG@20=0.1129, mới NDCG@20=0.0494 (119.7s)
  [Seed 2023] ep=12 | val_NDCG@20=0.0939 | (a) R@20=0.1667, NDCG@20=0.0729 | (b) NDCG@20=0.0259 | trùng NDCG@20=0.1435, mới NDC